In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# ===== RUTAS =====
base_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
output_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H1")
output_dir.mkdir(parents=True, exist_ok=True)

# ===== CARGA =====
macro = pd.read_csv(base_dir / "Dataset Maestro.csv")

# ===== ASEGURAR TIPOS =====
macro["price"] = pd.to_numeric(macro["price"], errors="coerce")
macro["rating"] = pd.to_numeric(macro["rating"], errors="coerce")
macro["review_count"] = pd.to_numeric(macro["review_count"], errors="coerce")
macro["subcategory"] = macro["subcategory"].astype("string").str.strip()

# ===== LIMPIEZA =====
df = macro.dropna(subset=["subcategory", "price"]).copy()

# ===== PERCENTILES PARA GAMA (Alta 10%, Media 40%, Baja 50%) =====
# Alta: >= p90 | Media: >= p50 y < p90 | Baja: < p50
percentiles = (
    df.groupby("subcategory")["price"]
    .quantile([0.50, 0.90])
    .unstack()
    .rename(columns={0.50: "p50", 0.90: "p90"})
    .reset_index()
)

df = df.merge(percentiles, on="subcategory", how="left")

def asignar_gama(row):
    if pd.isna(row["price"]) or pd.isna(row["p50"]) or pd.isna(row["p90"]):
        return pd.NA
    if row["price"] >= row["p90"]:
        return "alta"
    elif row["price"] >= row["p50"]:
        return "media"
    else:
        return "baja"

df["price_tier"] = df.apply(asignar_gama, axis=1)

# ===== TOP 10% POPULARIDAD =====
q90_reviews = (
    df.groupby("subcategory")["review_count"]
    .quantile(0.90)
    .rename("q90_review_count")
    .reset_index()
)

df = df.merge(q90_reviews, on="subcategory", how="left")
df["top_10_popularity"] = df["review_count"] >= df["q90_review_count"]

# ===== TABLAS POR GAMA =====
h1_alta = df[df["price_tier"] == "alta"].copy()
h1_media = df[df["price_tier"] == "media"].copy()
h1_baja = df[df["price_tier"] == "baja"].copy()

# ===== RESUMEN =====
h1_summary = (
    df.groupby(["subcategory", "price_tier", "top_10_popularity"], dropna=False)
    .agg(
        n_products=("product_name", "count"),
        avg_rating=("rating", "mean"),
        avg_review_count=("review_count", "mean"),
        avg_price=("price", "mean"),
    )
    .reset_index()
)

# ===== GUARDAR =====
df.to_csv(output_dir / "h1_dataset_completo.csv", index=False, encoding="utf-8-sig")
h1_alta.to_csv(output_dir / "h1_gama_alta.csv", index=False, encoding="utf-8-sig")
h1_media.to_csv(output_dir / "h1_gama_media.csv", index=False, encoding="utf-8-sig")
h1_baja.to_csv(output_dir / "h1_gama_baja.csv", index=False, encoding="utf-8-sig")
h1_summary.to_csv(output_dir / "h1_resumen.csv", index=False, encoding="utf-8-sig")

print(f"✅ Archivos guardados en: {output_dir}")
print(h1_summary.head())

✅ Archivos guardados en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H1
  subcategory price_tier  top_10_popularity  n_products  avg_rating  \
0      Cascos       alta               True           4         NaN   
1      Cascos       baja               True          20         NaN   
2      Cascos      media               True          16         NaN   
3      Mandos       alta               True           5         NaN   
4      Mandos       baja               True          19         NaN   

   avg_review_count      avg_price  
0              39.0  186240.000000  
1              39.0   26522.000000  
2              39.0   78110.625000  
3             780.0  101590.000000  
4             780.0   23374.210526  


In [4]:
from pathlib import Path
import matplotlib.pyplot as plt

output_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H1")
output_dir.mkdir(parents=True, exist_ok=True)

for tier in ["alta", "media", "baja"]:
    sub = h1_summary[h1_summary["price_tier"] == tier].copy()

    fig, ax = plt.subplots(figsize=(12, max(4, 0.45 * len(sub))))
    ax.axis("off")

    table = ax.table(
        cellText=sub.values,
        colLabels=sub.columns,
        cellLoc="center",
        loc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.3)

    plt.tight_layout()
    fig.savefig(output_dir / f"h1_resumen_{tier}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

print(f"PNG guardadas en: {output_dir}")

PNG guardadas en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H1
